# nb_04_silver_policies

**Layer**: Silver | **Source**: `lh_bronze_insurance.policies_raw` | **Target**: `lh_silver_insurance.policies`

**Data Quality Rules**:
- Cast `effective_date`, `expiration_date` to DateType; `annual_premium` to DoubleType
- Require non-null: `policy_id`, `customer_id`, `policy_type`
- Deduplicate on `policy_id` (keep latest `_ingested_at`)
- Standardize `status` and `policy_type` to lowercase
- Rejects → `policies_quarantine`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit, row_number, when, coalesce, lower
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_04_silver_policies").getOrCreate()

print("nb_04_silver_policies - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("policies_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & clean
# ============================================================
df_typed = df_bronze \
    .withColumn("policy_id", trim(col("policy_id"))) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("agent_id", trim(col("agent_id"))) \
    .withColumn("policy_type", lower(trim(col("policy_type")))) \
    .withColumn("effective_date", to_date(col("effective_date"), "yyyy-MM-dd")) \
    .withColumn("expiration_date", to_date(col("expiration_date"), "yyyy-MM-dd")) \
    .withColumn("status", lower(trim(col("status")))) \
    .withColumn("annual_premium", col("annual_premium").cast(DoubleType()))

print("Type casting complete.")

In [ ]:
# ============================================================
# Step 3: Validate & route rejects
# ============================================================
REQUIRED_FIELDS = ["policy_id", "customer_id", "policy_type"]

rejection_conditions = [
    when((col(f).isNull()) | (col(f) == ""), lit(f"null_{f}"))
    for f in REQUIRED_FIELDS
]

df_validated = df_typed.withColumn("_rejection_reason", coalesce(*rejection_conditions))

df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid: {valid_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on policy_id
# ============================================================
window_spec = Window.partitionBy("policy_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write Silver & Quarantine
# ============================================================
silver_columns = ["policy_id", "customer_id", "agent_id", "policy_type",
                  "effective_date", "expiration_date", "status", "annual_premium"]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("policies")
print(f"✓ {deduped_count:,} rows → policies")

if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("policies_quarantine")
    print(f"✓ {rejected_count:,} rows → policies_quarantine")

In [ ]:
# ============================================================
# Validation
# ============================================================
print("\nSILVER POLICIES - SUMMARY")
print("=" * 50)
print(f"  Bronze input:       {bronze_count:>8,}")
print(f"  Rejected:           {rejected_count:>8,}")
print(f"  Duplicates removed: {dupes_removed:>8,}")
print(f"  Silver output:      {deduped_count:>8,}")
print(f"  Quality rate:       {(deduped_count/bronze_count*100):.1f}%")

assert spark.table("policies").count() == deduped_count
print("\n✓ Verification passed")
spark.table("policies").show(5, truncate=25)